# Cálculo de la Intersección de Curvas No Lineales en el Plano 2D mediante Broyden



### Caso 1: Detección de Colisión / Intersección Geométrica
* **Objetivo:** Encontrar las coordenadas $(x, y)$ donde dos entidades geométricas colisionan simultáneamente.
* **Entrada (Input):** Estimación inicial arbitraria `x0 = [1.8, 0.8]`.
* **Criterio de parada:** Residuo $\|F(x)\| < 10^{-5}$ (distancia al cruce exacto).
* **Ventaja de Broyden:** No requiere calcular matrices de derivadas simbólicas; triangula la raíz solo evaluando residuos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. Definición del sistema geométrico: Círculo y Parábola
def F(p):
    x, y = p[0], p[1]
    return np.array([
        x**2 + y**2 - 4.0,  # Círculo: x^2 + y^2 = 4
        y - x**2            # Parábola: y = x^2
    ], dtype=float)

# 2. Jacobiano inicial por diferencias finitas (se calcula SOLO en el paso 0)
def jacobiano_inicial(F, x0, h=1e-5):
    n = len(x0)
    J = np.zeros((n, n))
    Fx0 = F(x0)
    for j in range(n):
        x_step = np.copy(x0)
        x_step[j] += h
        J[:, j] = (F(x_step) - Fx0) / h
    return J

# 3. Algoritmo de Broyden registrando la trayectoria paso a paso
def resolver_broyden_animado(x0, tol=1e-5, max_iter=10):
    x = np.array(x0, dtype=float)
    historia = [x.copy()]

    # Matriz inicial calculada una sola vez
    B = jacobiano_inicial(F, x)

    for _ in range(max_iter):
        Fx = F(x)
        if np.linalg.norm(Fx) < tol:
            break

        # Paso de dirección
        s = np.linalg.solve(B, -Fx)
        x_new = x + s

        # Medición del cambio real sin recalcular derivadas
        y_diff = F(x_new) - Fx

        # Actualización de Broyden
        B = B + np.outer(y_diff - B @ s, s) / np.dot(s, s)

        x = x_new
        historia.append(x.copy())


    return np.array(historia)

# Punto de partida (visible dentro del rango)
trayectoria = resolver_broyden_animado([1.8, 0.8])

# 4. Configuración del lienzo y curvas
fig, ax = plt.subplots(figsize=(7, 6))
x_grid = np.linspace(0.0, 2.4, 300)
y_grid = np.linspace(0.0, 2.4, 300)
X, Y = np.meshgrid(x_grid, y_grid)

# Curvas objetivo F1=0 y F2=0
ax.contour(X, Y, X**2 + Y**2 - 4, levels=[0], colors='#2563eb', linewidths=2.5)
ax.contour(X, Y, Y - X**2, levels=[0], colors='#d97706', linewidths=2.5)

# Elementos ficticios para la leyenda
ax.plot([], [], color='#2563eb', lw=2.5, label=r'Círculo: $x^2 + y^2 = 4$')
ax.plot([], [], color='#d97706', lw=2.5, label=r'Parábola: $y = x^2$')

# Elementos dinámicos que se animan
linea_camino, = ax.plot([], [], 'k--', lw=1.5, alpha=0.6, label='Trayectoria Broyden')
puntos_previos, = ax.plot([], [], 'ko', markersize=5, alpha=0.4)
punto_actual, = ax.plot([], [], 'ro', markersize=9, zorder=5, label='Aproximación actual')
texto_info = ax.text(0.04, 0.88, '', transform=ax.transAxes, fontsize=10,
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#cbd5e1'))

# Límites bien definidos para que todo quede centrado
ax.set_xlim(0.0, 2.3)
ax.set_ylim(0.0, 2.3)
ax.set_xlabel('Eje X', fontsize=11)
ax.set_ylabel('Eje Y', fontsize=11)
ax.set_title('Método de Broyden: Convergencia a la Intersección', fontsize=12, fontweight='bold')
ax.legend(loc='lower left', framealpha=0.9)
ax.grid(True, linestyle=':', alpha=0.6)

# 5. Función de actualización cuadro a cuadro
def update(frame):
    pts = trayectoria[:frame+1]
    linea_camino.set_data(pts[:, 0], pts[:, 1])
    puntos_previos.set_data(pts[:, 0], pts[:, 1])
    punto_actual.set_data([trayectoria[frame, 0]], [trayectoria[frame, 1]])

    err = np.linalg.norm(F(trayectoria[frame]))
    texto_info.set_text(
        f"Iteración: {frame}\n"
        f"x = {trayectoria[frame, 0]:.4f}, y = {trayectoria[frame, 1]:.4f}\n"
        f"Error ||F(x)||: {err:.2e}"
    )
    return linea_camino, puntos_previos, punto_actual, texto_info

anim = FuncAnimation(fig, update, frames=len(trayectoria), interval=1000, blit=True)
plt.close(fig)

HTML(anim.to_jshtml())